# Model Export Demo

This notebook demonstrates how to train and export your model to various formats including:
- PyTorch (for HuggingFace Hub)
- ONNX (for cross-platform deployment)
- TorchScript (for production)
- CoreML (for iOS/macOS)
- Kaggle Datasets

## Setup Environment

First, let's ensure we have all the required dependencies.

In [ ]:
# Install required dependencies
!pip install -q torch transformers datasets accelerate optimum onnx onnxruntime
!pip install -q kaggle safetensors sentencepiece

# Install platform-specific dependencies
import platform
import sys

# Install CoreML tools on macOS
if platform.system() == "Darwin":
    !pip install -q coremltools
    if platform.machine() == "arm64":  # Apple Silicon
        !pip install -q tensorflow-macos tensorflow-metal
    else:  # Intel Mac
        !pip install -q tensorflow
else:  # Linux/Windows
    !pip install -q tensorflow
    
# Install for TFLite conversion
!pip install -q tf2onnx

In [ ]:
import os
import logging
import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Import our custom modules
from src.model.architecture import create_model_from_config
from src.model.export import ModelExporter, export_model
from src.utils.export_helpers import export_to_all_formats, check_export_compatibility
from src.utils.kaggle_export import prepare_for_kaggle, upload_to_kaggle
from src.utils.tokenization import get_tokenizer

# Check if we're running on macOS for CoreML support
IS_MACOS = platform.system() == "Darwin"
if IS_MACOS:
    print("Running on macOS - CoreML export will be available")
else:
    print("Not running on macOS - CoreML export will be simulated")

## 1. Create a Simplified Training Configuration

For this demonstration, we'll create a simplified configuration for a small transformer model that we can quickly train.

In [ ]:
config = {
    "project_name": "export_demo_model",
    "output_dir": "./output_demo",
    "seed": 42,
    
    "model": {
        "type": "decoder_only_transformer",
        "size": "tiny",  # Using a tiny model for quick demonstration
        "sizes": {
            "tiny": {
                "n_layers": 2,
                "n_heads": 2,
                "d_model": 128,
                "d_ff": 512,
                "max_seq_length": 256
            }
        },
        "dropout": 0.1,
        "attention": {
            "causal": True,
            "rotary_embedding": True
        },
        "architecture": {
            "position_embeddings": "rotary",
            "attention_type": "mha",
            "norm_type": "layer_norm",
            "normalization_strategy": "pre_norm",
            "ffn_type": "gelu",
            "use_flash_attention": False
        },
        "compile": {
            "enabled": False
        },
        "gradient_checkpointing": False
    },
    
    "tokenizer": {
        "type": "huggingface",
        "name": "gpt2",
        "vocab_size": 50257,
        "max_length": 256,
        "padding_side": "right",
        "truncation_side": "right",
        "add_bos_token": True,
        "add_eos_token": True
    },
    
    "training": {
        "active_stage": "demo",
        "stages": [
            {
                "name": "demo",
                "epochs": 1,
                "datasets": [
                    {
                        "name": "imdb",
                        "split": "train",
                        "streaming": False,
                        "max_samples": 1000  # Just a small sample for demo
                    },
                    {
                        "name": "imdb",
                        "split": "test",
                        "streaming": False,
                        "max_samples": 100
                    }
                ],
                "learning_rate": {
                    "initial": 5.0e-4,
                    "min": 1.0e-6,
                    "schedule": "cosine",
                    "warmup_steps": 10
                }
            }
        ],
        "optimizer": {
            "name": "adamw",
            "weight_decay": 0.01,
            "beta1": 0.9,
            "beta2": 0.999,
            "eps": 1.0e-8,
            "use_8bit": False
        },
        "mixed_precision": "no",  # No mixed precision for simplicity
        "gradient_clipping": 1.0,
        "checkpointing": {
            "save_steps": 100,
            "keep_last_n": 1,
            "save_optimizer_state": True
        },
        "evaluation": {
            "eval_steps": 50
        }
    },
    
    "data_processing": {
        "preprocessing": {
            "normalize_unicode": True,
            "normalize_whitespace": True,
            "remove_html": True,
            "min_length": 10,
            "max_length": 256
        },
        "batching": {
            "train_batch_size": 8,
            "eval_batch_size": 8,
            "gradient_accumulation_steps": 1,
            "dynamic_batching": False
        }
    },
    
    "evaluation": {
        "metrics": [
            "loss",
            "perplexity",
            "accuracy"
        ],
        "generation": {
            "max_length": 64,
            "temperature": 0.8,
            "top_p": 0.92,
            "top_k": 50,
            "do_sample": True
        }
    },
    
    # Export configuration
    "export": {
        "formats": [
            "pytorch",
            "onnx",
            "torchscript",
            "coreml"  # Will be skipped automatically if not on macOS
        ],
        "metadata": {
            "author": "ModelExporter",
            "description": "Transformer model trained on IMDB dataset for demonstration",
            "license": "MIT",
            "framework": "PyTorch"
        },
        "huggingface_hub": {
            "enabled": False,
            "model_id": None,  # Format: "username/modelname"
            "private": True
        },
        "kaggle": {
            "enabled": False,
            "dataset_name": None,  # Format: "username/dataset-name"
            "is_public": False
        }
    }
}

# Save configuration to file
os.makedirs(config["output_dir"], exist_ok=True)
config_path = os.path.join(config["output_dir"], "config.yaml")
with open(config_path, "w") as f:
    yaml.dump(config, f)

print(f"Configuration saved to {config_path}")

## 2. Create Tokenizer and Model

Now let's create the tokenizer and model based on our configuration.

In [ ]:
# Get tokenizer
tokenizer = get_tokenizer(config["tokenizer"])
print(f"Loaded tokenizer with vocabulary size: {len(tokenizer)}")

# Create model
model = create_model_from_config(config)
print(f"Created model with {sum(p.numel() for p in model.parameters()):,} parameters")

# Set random seed for reproducibility
torch.manual_seed(config["seed"])
np.random.seed(config["seed"])

## 3. Train the Model

For this demonstration, we'll do a very quick training run on a small subset of data.

In [ ]:
# Load and prepare dataset
def load_and_prepare_dataset(config):
    """Load and prepare dataset for training."""
    # Get dataset config
    dataset_config = config["training"]["stages"][0]["datasets"][0]
    
    # Load dataset
    dataset = load_dataset(
        dataset_config["name"],
        split=dataset_config["split"]
    )
    
    # Limit samples if specified
    if dataset_config.get("max_samples") and len(dataset) > dataset_config["max_samples"]:
        dataset = dataset.select(range(dataset_config["max_samples"]))
    
    # Check for text column - IMDB uses 'text'
    text_column = "text"
    if text_column not in dataset.column_names:
        # Try to find a text column
        text_columns = [col for col in dataset.column_names if col in ["text", "content", "review"]]
        if text_columns:
            text_column = text_columns[0]
    
    # Define tokenization function
    def tokenize_function(examples):
        return tokenizer(
            examples[text_column],
            padding="max_length",
            truncation=True,
            max_length=config["tokenizer"]["max_length"]
        )
    
    # Tokenize dataset
    tokenized_dataset = dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=[col for col in dataset.column_names if col != text_column]
    )
    
    # Add labels for language modeling
    tokenized_dataset = tokenized_dataset.map(
        lambda examples: {"labels": examples["input_ids"]},
        batched=True
    )
    
    return tokenized_dataset

# Simplified trainer
class SimpleTrainer:
    def __init__(self, model, tokenizer, dataset, config):
        self.model = model
        self.tokenizer = tokenizer
        self.dataset = dataset
        self.config = config
        
        # Extract training params
        self.batch_size = config["data_processing"]["batching"]["train_batch_size"]
        self.epochs = config["training"]["stages"][0]["epochs"]
        self.lr = config["training"]["stages"][0]["learning_rate"]["initial"]
        
        # Setup optimizer
        self.optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=self.lr,
            weight_decay=config["training"]["optimizer"]["weight_decay"]
        )
        
        # Setup device
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
    
    def train(self):
        """Train the model."""
        # Set model to training mode
        self.model.train()
        
        # Create dataloader
        from torch.utils.data import DataLoader
        dataloader = DataLoader(
            self.dataset,
            batch_size=self.batch_size,
            shuffle=True
        )
        
        # Training loop
        total_steps = len(dataloader) * self.epochs
        progress_bar = range(total_steps)
        try:
            from tqdm.notebook import tqdm
            progress_bar = tqdm(progress_bar, desc="Training")
        except ImportError:
            pass
        
        step = 0
        for epoch in range(self.epochs):
            for batch in dataloader:
                # Move batch to device
                batch = {k: v.to(self.device) for k, v in batch.items()}
                
                # Forward pass
                outputs = self.model(**batch)
                loss = outputs.loss
                
                # Backward pass
                loss.backward()
                
                # Update parameters
                self.optimizer.step()
                self.optimizer.zero_grad()
                
                # Update progress bar
                if hasattr(progress_bar, "set_postfix"):
                    progress_bar.set_postfix({"loss": loss.item()})
                progress_bar.update(1)
                
                step += 1
                
                # Save checkpoint occasionally
                if step % 100 == 0:
                    checkpoint_path = os.path.join(self.config["output_dir"], "checkpoint.pt")
                    torch.save(self.model.state_dict(), checkpoint_path)
        
        # Save final model
        final_path = os.path.join(self.config["output_dir"], "final_model.pt")
        torch.save(self.model.state_dict(), final_path)
        
        return {"loss": loss.item()}

# Load and prepare dataset
train_dataset = load_and_prepare_dataset(config)
print(f"Prepared dataset with {len(train_dataset)} examples")

# Train model
trainer = SimpleTrainer(model, tokenizer, train_dataset, config)
results = trainer.train()
print(f"Training completed with final loss: {results['loss']:.4f}")

## 4. Test the Model

Let's verify that our model works by generating some text.

In [ ]:
# Set model to evaluation mode
model.eval()

# Function to generate text
def generate_text(prompt, max_length=50):
    # Tokenize prompt
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Generate text
    with torch.no_grad():
        outputs = model.generate(
            inputs["input_ids"],
            max_length=max_length,
            temperature=0.8,
            top_p=0.92,
            top_k=50,
            do_sample=True
        )
    
    # Decode and return text
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test with a few prompts
test_prompts = [
    "This movie was really",
    "I enjoyed watching",
    "The acting in this film"
]

for prompt in test_prompts:
    generated_text = generate_text(prompt)
    print(f"Prompt: {prompt}")
    print(f"Generated: {generated_text}")
    print()

## 5. Export the Model to Multiple Formats

Now let's export our trained model to various formats for deployment and sharing.

In [ ]:
# Create export directory
export_dir = os.path.join(config["output_dir"], "exported_models")
os.makedirs(export_dir, exist_ok=True)

# Check which formats are compatible with our model
compatibility = check_export_compatibility(model)
print("Model export compatibility:")
for fmt, compatible in compatibility.items():
    print(f"  {fmt}: {'✅' if compatible else '❌'}")

# Define input shape for tracing
input_shape = (1, config["tokenizer"]["max_length"])

# Create metadata
metadata = {
    "name": config["project_name"],
    "description": "Transformer model trained on IMDB dataset",
    "author": "Model Export Demo",
    "framework": "PyTorch",
    "task": "text-generation",
    "parameters": sum(p.numel() for p in model.parameters()),
    "architecture": {
        "type": config["model"]["type"],
        "layers": config["model"]["sizes"][config["model"]["size"]]["n_layers"],
        "heads": config["model"]["sizes"][config["model"]["size"]]["n_heads"],
        "dim": config["model"]["sizes"][config["model"]["size"]]["d_model"],
        "max_length": config["tokenizer"]["max_length"]
    }
}

# Define export formats
formats = ["pytorch", "onnx", "torchscript"]
if IS_MACOS:
    formats.append("coreml")

# Export model to all formats
exporter = ModelExporter(
    model=model,
    tokenizer=tokenizer,
    output_dir=export_dir,
    config=config
)

export_results = exporter.export_all(
    input_shapes={"input_ids": list(input_shape)},
    output_names=["logits"],
    formats=formats,
    metadata=metadata
)

print("\nExport completed:")
for fmt, path in export_results.items():
    print(f"  {fmt}: {path}")

## 6. Verify Exported Models

Let's make sure that our exported models work as expected.

In [ ]:
# Test PyTorch model
print("Testing PyTorch model:")
pytorch_dir = os.path.join(export_dir, "pytorch")
if os.path.exists(pytorch_dir):
    try:
        from transformers import AutoModelForCausalLM
        
        # Load model and tokenizer
        loaded_model = AutoModelForCausalLM.from_pretrained(pytorch_dir)
        loaded_tokenizer = AutoTokenizer.from_pretrained(pytorch_dir)
        
        # Test generation
        prompt = "This movie was"
        inputs = loaded_tokenizer(prompt, return_tensors="pt")
        outputs = loaded_model.generate(**inputs, max_length=20)
        generated_text = loaded_tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        print(f"  Input: {prompt}")
        print(f"  Output: {generated_text}")
        print("  ✅ PyTorch model works!")
    except Exception as e:
        print(f"  ❌ Error testing PyTorch model: {e}")
else:
    print("  ❌ PyTorch model not found")

# Test ONNX model
print("\nTesting ONNX model:")
onnx_dir = os.path.join(export_dir, "onnx")
if os.path.exists(onnx_dir):
    try:
        import onnxruntime as ort
        import numpy as np
        
        # Load ONNX session
        onnx_path = os.path.join(onnx_dir, "model.onnx")
        if not os.path.exists(onnx_path):
            onnx_path = os.path.join(onnx_dir, "model_optimized.onnx")
            
        session = ort.InferenceSession(onnx_path)
        
        # Create sample input
        input_ids = np.array([[101, 2023, 3232, 2001, 102, 0, 0, 0]], dtype=np.int64)  # Sample input
        
        # Run inference
        outputs = session.run(None, {"input_ids": input_ids})
        
        print(f"  Input shape: {input_ids.shape}")
        print(f"  Output shape: {outputs[0].shape}")
        print("  ✅ ONNX model works!")
    except Exception as e:
        print(f"  ❌ Error testing ONNX model: {e}")
else:
    print("  ❌ ONNX model not found")

# Test TorchScript model
print("\nTesting TorchScript model:")
torchscript_dir = os.path.join(export_dir, "torchscript")
if os.path.exists(torchscript_dir):
    try:
        import torch
        
        # Find TorchScript model file
        torchscript_path = os.path.join(torchscript_dir, "model.pt")
        if not os.path.exists(torchscript_path):
            torchscript_files = list(Path(torchscript_dir).glob("*.pt"))
            if torchscript_files:
                torchscript_path = str(torchscript_files[0])
        
        # Load TorchScript model
        loaded_model = torch.jit.load(torchscript_path)
        
        # Create sample input
        input_ids = torch.tensor([[101, 2023, 3232, 2001, 102, 0, 0, 0]], dtype=torch.long)
        
        # Run inference
        with torch.no_grad():
            outputs = loaded_model(input_ids)
        
        print(f"  Input shape: {input_ids.shape}")
        print(f"  Output shape: {outputs.shape}")
        print("  ✅ TorchScript model works!")
    except Exception as e:
        print(f"  ❌ Error testing TorchScript model: {e}")
else:
    print("  ❌ TorchScript model not found")

# Test CoreML model (macOS only)
if IS_MACOS:
    print("\nTesting CoreML model:")
    coreml_dir = os.path.join(export_dir, "coreml")
    if os.path.exists(coreml_dir):
        try:
            import coremltools as ct
            
            # Find CoreML model file
            coreml_path = os.path.join(coreml_dir, "model.mlmodel")
            if not os.path.exists(coreml_path):
                coreml_path = os.path.join(coreml_dir, "model.mlpackage")
            
            # Load CoreML model
            loaded_model = ct.models.MLModel(coreml_path)
            
            # Print model details
            print(f"  Input description: {loaded_model.input_description}")
            print(f"  Output description: {loaded_model.output_description}")
            print("  ✅ CoreML model loaded successfully!")
            
            # Note: Full prediction requires more setup for CoreML
        except Exception as e:
            print(f"  ❌ Error testing CoreML model: {e}")
    else:
        print("  ❌ CoreML model not found")
else:
    print("\nSkipping CoreML test (not on macOS)")

## 7. Prepare for Kaggle and HuggingFace

Now let's prepare our model for sharing on Kaggle and HuggingFace.

In [ ]:
# 7.1. Prepare for Kaggle Dataset
print("Preparing model for Kaggle...")
kaggle_dir = os.path.join(config["output_dir"], "kaggle_dataset")

# Prepare for Kaggle
from src.utils.kaggle_export import prepare_for_kaggle

kaggle_dataset_name = "your-username/transformer-model-demo"  # Change this to your Kaggle username
kaggle_description = f"""A small transformer model trained on IMDB dataset for text generation.

## Model Details
- Type: {config['model']['type']}
- Layers: {config['model']['sizes'][config['model']['size']]['n_layers']}
- Heads: {config['model']['sizes'][config['model']['size']]['n_heads']}
- Dimensions: {config['model']['sizes'][config['model']['size']]['d_model']}
- Parameters: {sum(p.numel() for p in model.parameters()):,}
"""

kaggle_path = prepare_for_kaggle(
    model_dir=export_dir,
    output_dir=kaggle_dir,
    dataset_name=kaggle_dataset_name,
    description=kaggle_description,
    keywords=["transformer", "nlp", "text-generation", "pytorch"],
    include_formats=["pytorch", "onnx", "torchscript"],
    max_size_mb=500
)

print(f"Kaggle dataset prepared at {kaggle_path}")
print("To upload to Kaggle, run:")
print("```")
print(f"kaggle datasets create -p {kaggle_path}")
print("```")

# 7.2. Prepare for HuggingFace Hub
print("\nPreparing model for HuggingFace Hub...")
# For HuggingFace, we typically just use the PyTorch format
hf_dir = os.path.join(export_dir, "pytorch")

# Add model card if it doesn't exist
model_card_path = os.path.join(hf_dir, "README.md")
if not os.path.exists(model_card_path):
    with open(model_card_path, "w") as f:
        f.write(f"""# {config['project_name']}

This is a small transformer model trained on IMDB dataset for text generation.

## Model Details
- Type: {config['model']['type']}
- Layers: {config['model']['sizes'][config['model']['size']]['n_layers']}
- Heads: {config['model']['sizes'][config['model']['size']]['n_heads']}
- Dimensions: {config['model']['sizes'][config['model']['size']]['d_model']}
- Parameters: {sum(p.numel() for p in model.parameters()):,}

## Usage

```python
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained("your-username/{config['project_name']}")
tokenizer = AutoTokenizer.from_pretrained("your-username/{config['project_name']}")

# Generate text
inputs = tokenizer("This movie was", return_tensors="pt")
outputs = model.generate(**inputs, max_length=50)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
```
""")

print(f"HuggingFace Hub model prepared at {hf_dir}")
print("To upload to HuggingFace Hub, run:")
print("```")
print(f"huggingface-cli login")
print(f"huggingface-cli upload your-username/{config['project_name']} {hf_dir}")
print("```")
print("\nOr in Python:")
print("```python")
print("from huggingface_hub import login, HfApi")
print("login()  # Will prompt for token")
print("api = HfApi()")
print(f"api.create_repo('your-username/{config['project_name']}', repo_type='model', exist_ok=True)")
print(f"api.upload_folder(folder_path='{hf_dir}', repo_id='your-username/{config['project_name']}')")
print("```")

## 8. Working with CoreML Models on macOS

If you're on macOS, here's how you can work with the exported CoreML model in more detail.

In [ ]:
if IS_MACOS:
    print("Working with CoreML model on macOS:")
    coreml_dir = os.path.join(export_dir, "coreml")
    if os.path.exists(coreml_dir):
        try:
            import coremltools as ct
            
            # Find CoreML model file
            coreml_path = os.path.join(coreml_dir, "model.mlmodel")
            if not os.path.exists(coreml_path):
                coreml_path = os.path.join(coreml_dir, "model.mlpackage")
            
            # Load CoreML model
            loaded_model = ct.models.MLModel(coreml_path)
            
            # Print model details
            print(f"\nModel specification:")
            print(f"  Inputs: {loaded_model.get_spec().description.input}")
            print(f"  Outputs: {loaded_model.get_spec().description.output}")
            
            # Get model metadata
            metadata = loaded_model.user_defined_metadata
            print(f"\nModel metadata:")
            for key, value in metadata.items():
                print(f"  {key}: {value}")
            
            # Show example Swift code for using the model
            print(f"\nExample Swift code for iOS/macOS integration:")
            swift_code = """import CoreML

// Load the CoreML model
guard let modelURL = Bundle.main.url(forResource: "model", withExtension: "mlmodel") else {
    fatalError("Failed to find model in bundle")
}

let model: MLModel
do {
    let compiledModelURL = try MLModel.compileModel(at: modelURL)
    model = try MLModel(contentsOf: compiledModelURL)
} catch {
    fatalError("Failed to load CoreML model: \(error)")
}

// Prepare input
let inputName = "input_ids"
let shape: [NSNumber] = [1, 32]  // [batch_size, sequence_length]
let inputArray = [Int32](repeating: 0, count: shape.reduce(1, *))

// Create MLMultiArray for input
let inputMultiArray: MLMultiArray
do {
    inputMultiArray = try MLMultiArray(shape: shape, dataType: .int32)
    
    // Fill with data (example input_ids)
    for (index, value) in inputArray.enumerated() {
        inputMultiArray[index] = NSNumber(value: value)
    }
} catch {
    fatalError("Failed to create input MLMultiArray: \(error)")
}

// Create model input
let input = try! MLDictionaryFeatureProvider(dictionary: [inputName: inputMultiArray])

// Get prediction
guard let output = try? model.prediction(from: input) else {
    fatalError("Failed to get prediction")
}

// Process the output
if let outputFeatures = output.featureValue(for: "logits") {
    // Process logits
    let logitsMultiArray = outputFeatures.multiArrayValue!
    
    // Use the logits for text generation
    // ...
}
"""
            print(swift_code)
            
            # Create info about converting to Core ML Tools 6+
            print(f"\nUsing CoreML Tools 6+ for Neural Networks:")
            print("For the latest CoreML Tools 6+ with Neural Network support:")
            print("1. Ensure you're using macOS 12+ (Monterey or later)")
            print("2. Install coremltools 6+: pip install coremltools>=6.0")
            print("3. Use Neural Network API for advanced operations:")
            print("```python")
            print("import coremltools as ct")
            print("import coremltools.neural_network as nn")
            print("```")
            
            # Testing on real devices
            print(f"\nTo test on real iOS devices:")
            print("1. Add the .mlmodel file to your Xcode project")
            print("2. Xcode will automatically generate Swift/Objective-C wrapper classes")
            print("3. Use the model in your app with the generated classes")
            
        except Exception as e:
            print(f"❌ Error examining CoreML model: {e}")
    else:
        print("❌ CoreML model not found")
else:
    print("CoreML detailed examination skipped (not on macOS)")

## 9. Summary

We've successfully completed the following steps:

1. Created and trained a small transformer model
2. Tested the model to ensure it works properly
3. Exported the model to multiple formats:
   - PyTorch (for HuggingFace Hub)
   - ONNX (for cross-platform deployment)
   - TorchScript (for production)
   - CoreML (for iOS/macOS, if on macOS)
4. Verified that the exported models work
5. Prepared the model for sharing on Kaggle and HuggingFace

This workflow demonstrates how to take a model from training to deployment across different platforms and formats.